# Negative Feedback — News Recommender Extension

This notebook extends `InteractiveRecommender.ipynb` by adding **negative feedback**: a mechanism that lets the system learn not just from what the user *clicks*, but also from what they *skip*.

All existing functions are kept **untouched**. The new code is added as an extra layer on top.

---
## Section 1 — What is the Problem with the Current System?

Right now, your recommender only learns from **positive signals** — articles the user clicks.

But every round, 4 out of 5 articles are shown and **skipped**. The system currently treats that silence as if it never happened.

**Example:**

> Round 1: You're shown 5 travel articles. You click article 3 (a beach holiday piece).  
> The other 4 (ski resorts, camping guides, road trips, adventure travel) were shown to you but you ignored them.  
> The system learns: *"user likes beaches"*.  
> What it **doesn't** learn: *"user is not particularly interested in ski resorts, camping, or road trips"*.

Over many rounds, this causes two problems:
1. **Slow convergence** — the profile drifts toward what you like, but never away from what you don't.
2. **Echo chamber risk** — topics you consistently skip keep reappearing because there's no signal telling the system to reduce them.

---
## Section 2 — What is Negative Feedback?

Negative feedback means: **if an article was shown to you but you didn't click it, the system should slightly move your profile away from that article's content**.

Think of it like tuning a radio:
- Clicking an article = you turned the dial *toward* that frequency.
- Skipping an article = you turned the dial *slightly away* from that frequency.

The key word is **slightly**. Skipping is a much weaker signal than clicking. You might skip an article because:
- The headline didn't catch your eye (even if you'd enjoy the content)
- You've already read something similar
- You were in a hurry

So we apply a small penalty, not a large one. This is controlled by the `negative_weight` parameter.

---
## Section 3 — How Does it Work Mathematically?

Your user profile is a **vector of 5,000 numbers** (one for each TF-IDF feature/word). Each number represents how strongly the user is associated with that word/concept.

Every article also has a TF-IDF vector of 5,000 numbers. When you click an article, its vector gets added (with weight) to your profile — pulling the profile toward that article's topic.

**Negative feedback does the opposite:**

```
updated_profile = current_profile  −  (negative_weight × skipped_article_vector)
```

So if a ski resort article has a high value for the word "ski", and you skip it, the "ski" dimension in your profile gets slightly reduced.

**Step by step for one round:**

```
Round 2 shown articles:  [ski resort, camping, road trip, beach holiday, adventure]
User clicks:              beach holiday  ← positive signal
User skips:               ski resort, camping, road trip, adventure  ← negative signal

1. Rebuild profile from all positive clicks (as before)
2. For each skipped article:
       profile = profile − (0.05 × skipped_vector)
3. Clip any negative values to 0
       (a dimension can't go below zero — you can't have "negative interest")
```

After step 3, the profile is slightly less associated with "ski", "camping", "road trip", "adventure" — and will score those articles lower next round.

---
## Section 4 — The `negative_weight` Parameter

The `negative_weight` controls how strongly a skip penalises the profile.

| Value | Effect |
|-------|--------|
| `0.00` | No negative feedback (same as current system) |
| `0.02` | Very gentle nudge — safe, barely noticeable |
| `0.05` | **Default — recommended starting point** |
| `0.10` | Strong penalty — profile shifts noticeably after 3–4 skips |
| `0.20+` | Aggressive — risky, can over-correct and distort the profile |

**Why keep it small?**

Because a skip is an ambiguous signal. We don't know *why* the user skipped — maybe they just didn't see a good headline. We never want a few skips to completely erase a category from recommendations. A small `negative_weight` accumulates over many rounds naturally without over-reacting to any single round.

**Why do we clip to 0?**

TF-IDF values are always ≥ 0 (they represent presence of words, not absence). If we subtract too aggressively, some dimensions could go negative, which has no meaningful interpretation. Clipping ensures the profile stays in a valid range.

---
## Section 5 — What to Expect When You Run It

Compared to the original system:

- **Faster topic focus** — If you consistently skip sports articles, sports will fade from recommendations sooner.
- **Less repetition** — Articles very similar to ones you've skipped will score lower, so you won't keep seeing the same type of headline.
- **More responsive pivots** — If you start skipping travel and clicking finance, the profile shifts faster than with clicks alone.

The difference is **subtle per round** but **cumulative over a session**. After 5+ rounds, profiles with negative feedback will diverge noticeably from profiles without it.

---
## Code Starts Below

In [2]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import vstack
from collections import Counter

# Load news dataset
news_df = pd.read_csv(
    "../MINDsmall_train/news.tsv",
    sep="\t",
    header=None,
    names=[
        "news_id", "category", "subcategory",
        "title", "abstract", "url",
        "title_entities", "abstract_entities"
    ]
)

news_df["title"]    = news_df["title"].fillna("")
news_df["abstract"] = news_df["abstract"].fillna("")
news_df["url"]      = news_df["url"].fillna("")
news_df = news_df[["news_id", "category", "subcategory", "title", "abstract", "url"]]
news_df["content"]  = news_df["title"] + " " + news_df["abstract"]

# Load behaviors
behaviors_df = pd.read_csv(
    "../MINDsmall_train/behaviors.tsv",
    sep="\t",
    header=None,
    names=["impression_id", "user_id", "time", "history", "impressions"]
)
behaviors_df["history"] = behaviors_df["history"].fillna("").apply(lambda x: x.split())

# TF-IDF
tfidf        = TfidfVectorizer(stop_words="english", max_features=5000)
tfidf_matrix = tfidf.fit_transform(news_df["content"])

# Mappings
news_id_to_index       = {nid: idx for idx, nid in enumerate(news_df["news_id"])}
news_df["recency_score"] = news_df.index / len(news_df)

print(f"News loaded       : {len(news_df):,} articles")
print(f"Behaviors loaded  : {len(behaviors_df):,} records")
print(f"TF-IDF matrix     : {tfidf_matrix.shape}")

News loaded       : 51,282 articles
Behaviors loaded  : 156,965 records
TF-IDF matrix     : (51282, 5000)


In [3]:
# ── Existing functions (unchanged) ────────────────────────────────────

def build_user_profile(clicked_news_ids):
    """
    Weighted average of TF-IDF vectors for clicked articles.
    Recent clicks get higher weight: weight = (i+1) / total
    """
    vectors, weights = [], []
    total = len(clicked_news_ids)
    for i, news_id in enumerate(clicked_news_ids):
        if news_id in news_id_to_index:
            vectors.append(tfidf_matrix[news_id_to_index[news_id]])
            weights.append((i + 1) / total)
    if not vectors:
        return None
    stacked          = vstack(vectors)
    weights          = np.array(weights).reshape(-1, 1)
    weighted_profile = stacked.multiply(weights).sum(axis=0) / weights.sum()
    return np.asarray(weighted_profile)


def recommend_articles_with_recency(
    user_profile, clicked_news_ids,
    selected_category=None, selected_subcategory=None,
    alpha=0.9, beta=0.1, top_n=5, candidate_pool=100
):
    similarity_scores = cosine_similarity(user_profile, tfidf_matrix).flatten()
    sorted_indices    = similarity_scores.argsort()[::-1]
    recommendations   = []
    for idx in sorted_indices:
        row = news_df.iloc[idx]
        if row["news_id"] in clicked_news_ids: continue
        if selected_category    and row["category"]    != selected_category:    continue
        if selected_subcategory and row["subcategory"] != selected_subcategory: continue
        sim_score   = similarity_scores[idx]
        final_score = alpha * sim_score + beta * row["recency_score"]
        recommendations.append((idx, sim_score, row["recency_score"], final_score))
        if len(recommendations) >= candidate_pool: break
    recommendations = sorted(recommendations, key=lambda x: x[3], reverse=True)[:top_n]
    results = []
    for idx, sim, rec, final in recommendations:
        row = news_df.iloc[idx][["news_id","title","abstract","category","subcategory","url"]].to_dict()
        row.update({"similarity_score": round(sim,4), "recency_score": round(rec,4), "final_score": round(final,4)})
        results.append(row)
    return pd.DataFrame(results)


def create_user_profile_from_interest(category, subcategory, n_seed=5):
    seed_articles = news_df[
        (news_df["category"] == category) &
        (news_df["subcategory"] == subcategory)
    ].head(n_seed)
    clicked_ids  = seed_articles["news_id"].tolist()
    user_profile = build_user_profile(clicked_ids)
    return user_profile, clicked_ids


print("Core functions loaded.")

Core functions loaded.


In [4]:
def compute_popularity_scores(behaviors_df, news_df):
    click_counts = Counter()
    for impression_str in behaviors_df["impressions"]:
        for item in impression_str.split():
            news_id, label = item.split("-")
            if int(label) == 1:
                click_counts[news_id] += 1
    news_df["popularity_raw"]   = news_df["news_id"].map(click_counts).fillna(0)
    max_clicks                  = news_df["popularity_raw"].max()
    news_df["popularity_score"] = news_df["popularity_raw"] / max_clicks if max_clicks > 0 else 0
    return news_df

behaviors_raw = pd.read_csv(
    "../MINDsmall_train/behaviors.tsv",
    sep="\t", header=None,
    names=["impression_id","user_id","time","history","impressions"]
)
news_df = compute_popularity_scores(behaviors_raw, news_df)

print(f"Popularity scores computed.")
print(f"Most clicked article : {int(news_df['popularity_raw'].max())} clicks")
print(f"Articles with 0 clicks: {(news_df['popularity_raw'] == 0).sum():,}")

Popularity scores computed.
Most clicked article : 4316 clicks
Articles with 0 clicks: 43,569


In [5]:
# ── NEW: Negative Feedback Function ───────────────────────────────────

def apply_negative_feedback(user_profile, skipped_news_ids, negative_weight=0.05):
    """
    Adjusts the user profile AWAY from articles that were shown but not clicked.

    For each skipped article, subtracts a small fraction of its TF-IDF vector
    from the current user profile.  Clips to 0 afterward so no dimension
    goes negative (TF-IDF values are always >= 0).

    Parameters
    ----------
    user_profile      : np.array  — current profile vector (shape: 1 x 5000)
    skipped_news_ids  : list      — article IDs shown but not clicked this round
    negative_weight   : float     — how strongly to penalise skips (default 0.05)
                                    recommended range: 0.02 – 0.10

    Returns
    -------
    np.array — adjusted profile vector (same shape as input)
    """
    if not skipped_news_ids or user_profile is None:
        return user_profile

    # Work on a float copy so we don't mutate the original
    adjusted = user_profile.copy().astype(float)

    for news_id in skipped_news_ids:
        if news_id in news_id_to_index:
            idx            = news_id_to_index[news_id]
            skipped_vector = np.asarray(tfidf_matrix[idx].todense()).flatten()

            # Subtract: pull profile away from this article's content
            adjusted -= negative_weight * skipped_vector

    # Clip: interest cannot go below 0
    adjusted = np.clip(adjusted, 0, None)

    return adjusted


print("apply_negative_feedback() loaded.")
print()
print("Signature  : apply_negative_feedback(user_profile, skipped_news_ids, negative_weight=0.05)")
print("What it does:")
print("  1. Loops over each article the user was shown but did NOT click.")
print("  2. Subtracts (negative_weight × that article's TF-IDF vector) from the profile.")
print("  3. Clips every dimension to >= 0.")
print("  4. Returns the adjusted profile — ready to be used for the next round.")

apply_negative_feedback() loaded.

Signature  : apply_negative_feedback(user_profile, skipped_news_ids, negative_weight=0.05)
What it does:
  1. Loops over each article the user was shown but did NOT click.
  2. Subtracts (negative_weight × that article's TF-IDF vector) from the profile.
  3. Clips every dimension to >= 0.
  4. Returns the adjusted profile — ready to be used for the next round.


In [6]:
# ── UserSession class — extended with negative feedback ───────────────

class UserSession:
    """
    Manages a single user's live recommendation session.
    Extended: update() now optionally accepts skipped article IDs and
    applies negative feedback after rebuilding the profile.
    """

    def __init__(self, category, subcategory):
        self.selected_category     = category
        self.selected_subcategory  = subcategory
        self.round_number          = 0
        self.session_log           = []
        self.total_skipped         = 0       # NEW: tracks cumulative skips

        self.user_profile, self.clicked_ids = create_user_profile_from_interest(
            category, subcategory
        )

        print(f"\n Session started!")
        print(f" Category    : {category}")
        print(f" Subcategory : {subcategory}")
        print(f" Seed articles: {len(self.clicked_ids)}")


    def update(self, clicked_news_id, skipped_news_ids=None, negative_weight=0.05):
        """
        Call after user clicks an article.

        Steps:
        1. Add clicked article to history.
        2. Rebuild profile from full positive history (same as before).
        3. NEW: Apply negative feedback for all skipped articles.

        Parameters
        ----------
        clicked_news_id  : str   — the article the user clicked
        skipped_news_ids : list  — articles shown but not clicked (can be None)
        negative_weight  : float — strength of the negative signal (default 0.05)
        """
        # Step 1 — add click
        if clicked_news_id not in self.clicked_ids:
            self.clicked_ids.append(clicked_news_id)

        # Step 2 — rebuild from positive history
        self.user_profile = build_user_profile(self.clicked_ids)

        # Step 3 — apply negative feedback (NEW)
        if skipped_news_ids:
            self.total_skipped += len(skipped_news_ids)
            self.user_profile   = apply_negative_feedback(
                self.user_profile, skipped_news_ids, negative_weight
            )


    def get_dominant_category(self):
        clicked_rows = news_df[news_df["news_id"].isin(self.clicked_ids)]
        if clicked_rows.empty:
            return self.selected_category
        return clicked_rows["category"].value_counts().idxmax()


    def get_interest_summary(self):
        clicked_rows = news_df[news_df["news_id"].isin(self.clicked_ids)]
        return clicked_rows["category"].value_counts().to_dict()


    def log_round(self, shown_ids, clicked_id, skipped_ids):
        self.session_log.append({
            "round"      : self.round_number,
            "shown"      : shown_ids,
            "clicked"    : clicked_id,
            "skipped"    : skipped_ids,       # NEW: log skips too
            "profile_cat": self.get_dominant_category()
        })


print("UserSession class (with negative feedback) ready.")

UserSession class (with negative feedback) ready.


In [7]:
# ── Enhanced recommender + diversity (unchanged from InteractiveRecommender) ──

def recommend_with_popularity(
    user_profile, clicked_news_ids,
    selected_category=None, selected_subcategory=None,
    alpha=0.8, beta=0.1, gamma=0.1,
    top_n=5, candidate_pool=150
):
    similarity_scores = cosine_similarity(user_profile, tfidf_matrix).flatten()
    sorted_indices    = similarity_scores.argsort()[::-1]
    candidates        = []
    for idx in sorted_indices:
        row = news_df.iloc[idx]
        if row["news_id"] in clicked_news_ids: continue
        if selected_category    and row["category"]    != selected_category:    continue
        if selected_subcategory and row["subcategory"] != selected_subcategory: continue
        sim   = similarity_scores[idx]
        rec   = row["recency_score"]
        pop   = row["popularity_score"]
        final = alpha * sim + beta * rec + gamma * pop
        candidates.append((idx, sim, rec, pop, final))
        if len(candidates) >= candidate_pool: break
    candidates = sorted(candidates, key=lambda x: x[4], reverse=True)
    results = []
    for idx, sim, rec, pop, final in candidates[:top_n]:
        row = news_df.iloc[idx][["news_id","title","abstract","category","subcategory","url"]].to_dict()
        row.update({
            "similarity_score": round(sim,4), "recency_score": round(rec,4),
            "popularity_score": round(pop,4), "final_score":   round(final,4)
        })
        results.append(row)
    return pd.DataFrame(results)


def inject_diversity(recommendations_df, clicked_news_ids, n_diverse=1):
    if recommendations_df.empty:
        return recommendations_df
    dominant_sub = recommendations_df.iloc[0]["subcategory"]
    exclude_ids  = set(clicked_news_ids) | set(recommendations_df["news_id"])
    diverse_pool = news_df[
        (news_df["subcategory"] != dominant_sub) &
        (~news_df["news_id"].isin(exclude_ids))
    ].sort_values("popularity_score", ascending=False)
    if diverse_pool.empty:
        return recommendations_df
    diverse_articles = diverse_pool.head(n_diverse).copy()
    diverse_articles["similarity_score"] = 0.0
    diverse_articles["popularity_score"] = diverse_articles["popularity_score"]
    diverse_articles["final_score"]      = 0.0
    keep_cols = ["news_id","title","abstract","category","subcategory","url",
                 "similarity_score","recency_score","popularity_score","final_score"]
    recommendations_df = recommendations_df.copy()
    recommendations_df["is_diverse"] = False
    main_recs = recommendations_df.iloc[:len(recommendations_df) - n_diverse]
    return pd.concat([main_recs, diverse_articles[keep_cols].assign(is_diverse=True)],
                     ignore_index=True)


print("Enhanced recommender and diversity injection ready.")

Enhanced recommender and diversity injection ready.


In [8]:
# ── Display helpers — updated to show negative feedback info ──────────

def display_recommendations(recommendations_df, round_number):
    print(f"\n{'='*65}")
    print(f"  ROUND {round_number}  —  Top Recommendations")
    print(f"{'='*65}")
    for i, row in recommendations_df.iterrows():
        diverse_tag    = "  [EXPLORE]" if row.get("is_diverse", False) else ""
        title_short    = row["title"][:70] + "..." if len(row["title"]) > 70 else row["title"]
        abstract_short = str(row["abstract"])[:90] + "..." if len(str(row["abstract"])) > 90 else str(row["abstract"])
        print(f"\n  [{i+1}] {title_short}{diverse_tag}")
        print(f"       Category : {row['category']} > {row['subcategory']}")
        print(f"       Score    : {row['final_score']:.4f}  "
              f"(sim={row['similarity_score']:.3f}, "
              f"rec={row['recency_score']:.3f}, "
              f"pop={row['popularity_score']:.3f})")
        print(f"       Preview  : {abstract_short}")
    print(f"\n{'─'*65}")


def display_profile_drift(session, skipped_titles):
    """Updated to also show which articles were penalised this round."""
    summary  = session.get_interest_summary()
    dominant = session.get_dominant_category()
    print(f"\n  Profile update (with negative feedback):")
    print(f"  Total articles in history : {len(session.clicked_ids)}")
    print(f"  Total skips penalised     : {session.total_skipped}")  # NEW
    print(f"  Dominant interest         : {dominant}")

    # NEW: Show which articles were down-weighted this round
    if skipped_titles:
        print(f"\n  Down-weighted this round (skipped):")
        for t in skipped_titles:
            short = t[:60] + "..." if len(t) > 60 else t
            print(f"    ↓  {short}")

    print(f"\n  Interest breakdown:", end=" ")
    for cat, count in sorted(summary.items(), key=lambda x: x[1], reverse=True):
        print(f"\n    {cat:<20} {'█' * count} ({count})", end="")
    print()


def display_session_summary(session):
    print(f"\n{'='*65}")
    print(f"  SESSION SUMMARY")
    print(f"{'='*65}")
    print(f"  Rounds completed     : {session.round_number}")
    print(f"  Articles clicked     : {len(session.clicked_ids)}")
    print(f"  Total skips applied  : {session.total_skipped}")   # NEW
    print(f"  Started with         : {session.selected_category} > {session.selected_subcategory}")
    print(f"  Final dominant topic : {session.get_dominant_category()}")
    print(f"\n  Round-by-round history:")
    for entry in session.session_log:
        clicked_rows  = news_df[news_df["news_id"] == entry["clicked"]]
        clicked_title = ""
        if not clicked_rows.empty:
            t = clicked_rows.iloc[0]["title"]
            clicked_title = t[:50] + "..." if len(t) > 50 else t
        print(f"  Round {entry['round']:>2}  |  Clicked : {clicked_title}")
        print(f"           |  Skipped : {len(entry['skipped'])} articles penalised")
        print(f"           |  Dominant: {entry['profile_cat']}")
    print(f"\n  Categories explored:")
    for cat, count in sorted(session.get_interest_summary().items(), key=lambda x: x[1], reverse=True):
        print(f"    {cat:<25} {count} articles")
    print(f"\n{'='*65}")
    print("  Thanks for using the News Recommender (with Negative Feedback)!")
    print(f"{'='*65}\n")


print("Display helpers ready.")

Display helpers ready.


In [9]:
# ── run_session() — updated to collect skipped IDs and apply negative feedback ──

def run_session(
    top_n=5,
    diversity_slots=1,
    alpha=0.8,
    beta=0.1,
    gamma=0.1,
    negative_weight=0.05,    # NEW parameter
    use_popularity=True,
    filter_by_category=True
):
    """
    Full interactive recommendation session with negative feedback.

    New parameter
    -------------
    negative_weight : float — how strongly to penalise skipped articles (default 0.05).
                              Set to 0.0 to disable negative feedback entirely.
    """

    # ── Cold start ──────────────────────────────────────────────────────
    print("\n" + "="*65)
    print("  WELCOME — NEWS RECOMMENDER (with Negative Feedback)")
    print("="*65)
    print("\nAvailable Categories:\n")
    categories = sorted(news_df["category"].unique())
    for i, cat in enumerate(categories):
        print(f"  {i:>2}: {cat}")

    cat_index         = int(input("\nSelect category number: "))
    selected_category = categories[cat_index]

    subcats = sorted(news_df[news_df["category"] == selected_category]["subcategory"].unique())
    print(f"\nAvailable Subcategories for '{selected_category}':\n")
    for i, sub in enumerate(subcats):
        print(f"  {i:>2}: {sub}")

    sub_index            = int(input("\nSelect subcategory number: "))
    selected_subcategory = subcats[sub_index]

    session    = UserSession(selected_category, selected_subcategory)
    cat_filter = selected_category if filter_by_category else None

    # ── Main loop ───────────────────────────────────────────────────────
    while True:
        session.round_number += 1

        # Generate recommendations
        if use_popularity:
            recs = recommend_with_popularity(
                session.user_profile, session.clicked_ids,
                selected_category=cat_filter,
                alpha=alpha, beta=beta, gamma=gamma, top_n=top_n
            )
        else:
            recs = recommend_articles_with_recency(
                session.user_profile, session.clicked_ids,
                selected_category=cat_filter,
                alpha=alpha, beta=beta, top_n=top_n
            )

        if recs.empty:
            print("\n  Category pool exhausted — expanding to all categories...")
            cat_filter = None
            recs = recommend_with_popularity(
                session.user_profile, session.clicked_ids,
                alpha=alpha, beta=beta, gamma=gamma, top_n=top_n
            )

        if diversity_slots > 0 and len(recs) >= diversity_slots + 1:
            recs = inject_diversity(recs, session.clicked_ids, n_diverse=diversity_slots)

        display_recommendations(recs, session.round_number)

        # ── User input ──────────────────────────────────────────────────
        nw_status = f"negative_weight={negative_weight}" if negative_weight > 0 else "negative feedback OFF"
        print(f"  Enter 1–{len(recs)} to read  |  'q' to quit  |  [{nw_status}]")
        user_input = input("\n  Your choice: ").strip().lower()

        if user_input == "q":
            display_session_summary(session)
            break

        try:
            choice = int(user_input)
            if choice < 1 or choice > len(recs):
                print(f"\n  Please enter a number between 1 and {len(recs)}.")
                session.round_number -= 1
                continue
        except ValueError:
            print("\n  Invalid input. Please enter a number or 'q'.")
            session.round_number -= 1
            continue

        # ── Process click ───────────────────────────────────────────────
        clicked_row     = recs.iloc[choice - 1]
        clicked_news_id = clicked_row["news_id"]
        clicked_title   = clicked_row["title"]

        # ── NEW: Identify skipped articles ─────────────────────────────
        # All shown articles except the clicked one are "skipped"
        skipped_ids    = [r["news_id"] for _, r in recs.iterrows()
                          if r["news_id"] != clicked_news_id]
        skipped_titles = [r["title"]   for _, r in recs.iterrows()
                          if r["news_id"] != clicked_news_id]

        print(f"\n  You selected : {clicked_title[:70]}")
        print(f"  Category     : {clicked_row['category']} > {clicked_row['subcategory']}")
        print(f"  Penalising   : {len(skipped_ids)} skipped articles")

        # Log before updating
        session.log_round(
            shown_ids   = recs["news_id"].tolist(),
            clicked_id  = clicked_news_id,
            skipped_ids = skipped_ids        # NEW
        )

        # Update: rebuilds profile + applies negative feedback
        session.update(
            clicked_news_id,
            skipped_news_ids = skipped_ids,
            negative_weight  = negative_weight
        )

        display_profile_drift(session, skipped_titles)


print("run_session() ready. Run the next cell to start.")

run_session() ready. Run the next cell to start.


In [10]:
# ── Launch ────────────────────────────────────────────────────────────
# Tune negative_weight here:
#   0.00 = no negative feedback (same as original)
#   0.02 = very gentle
#   0.05 = default, recommended
#   0.10 = stronger signal

run_session(
    top_n           = 5,
    diversity_slots = 1,
    alpha           = 0.8,    # similarity weight
    beta            = 0.1,    # recency weight
    gamma           = 0.1,    # popularity weight
    negative_weight = 0.05,   # NEW: negative feedback strength
    use_popularity  = True,
    filter_by_category = True
)


  WELCOME — NEWS RECOMMENDER (with Negative Feedback)

Available Categories:

   0: autos
   1: entertainment
   2: finance
   3: foodanddrink
   4: health
   5: kids
   6: lifestyle
   7: middleeast
   8: movies
   9: music
  10: news
  11: northamerica
  12: sports
  13: travel
  14: tv
  15: video
  16: weather

Available Subcategories for 'entertainment':

   0: awards
   1: celebhub
   2: celebrity
   3: celebritynews
   4: entertainment-books
   5: entertainment-celebrity
   6: entertainmentmusic
   7: entertainmenttv
   8: games
   9: gaming
  10: hollywood
  11: humor
  12: news
  13: tv
  14: video

 Session started!
 Category    : entertainment
 Subcategory : hollywood
 Seed articles: 1

  ROUND 1  —  Top Recommendations

  [1] Miley Cyrus clarifies Instagram remarks about sexuality being a choice...
       Category : entertainment > entertainment-celebrity
       Score    : 0.2334  (sim=0.253, rec=0.311, pop=0.000)
       Preview  : Miley Cryus responds to criticism of Inst